In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from Bio import SeqIO
import os
from collections import Counter
import scipy
from scipy.stats import mannwhitneyu, chi2_contingency,ks_2samp,kruskal
from matplotlib.patches import Patch

plt.rcParams.update({'font.family':'Arial'})




In [58]:
base_dir = '/Users/cdubin/Box/VMGC_cervical_dysplasia_paper/code/'

In [59]:
species = pd.read_csv(f'{base_dir}/compare_VMGC_GTDB/select_species_for_analysis/shared_species_for_analysis.csv')
species = species.set_index('species_id_VMGC')
species_list = species.index.tolist()
species

,species,species_id_GTDB,num_samples_GTDB,num_samples_VMGC
species_id_VMGC,,,,
988598,Lactobacillus crispatus,100122,135,135
240891,Lactobacillus iners,100505,120,121
783244,Bifidobacterium vaginale,100323,78,78
619501,Fannyhessea vaginae,103895,65,43
571325,Lactobacillus jensenii,100515,35,35
611554,Lactobacillus gasseri,100460,20,20


In [60]:
df = pd.read_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/compare_VMGC_GTDB/combined_db/combined_db_analysis/genomes_info_source_and_genome_type.csv')

In [61]:
df['genome_no_version'] = df['genome'].apply(
    lambda x: x.split('.')[0] if str(x).startswith(('GCF', 'GCA')) else x
)
df['genome_no_version'] = df['genome_no_version'].str.replace(r'^GCA', 'GCF', regex=True)

In [62]:
counts = df['genome_no_version'].value_counts() 
df[df['genome_no_version'].isin(counts[counts>1].index.tolist())].sort_values('genome')

,genome,species,representative,genome_is_representative,database,genome_type,species_name,source,genome_no_version
4890,GCF_000191685.1,240891,SRR17284223.mbin.1,0,GTDB,isolate,Lactobacillus iners,FRT,GCF_000191685
78,GCF_000191685.2,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,GCF_000191685
4891,GCF_000191705.1,240891,SRR17284223.mbin.1,0,GTDB,isolate,Lactobacillus iners,FRT,GCF_000191705
119,GCF_000191705.2,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,GCF_000191705
4892,GCF_000204435.1,240891,SRR17284223.mbin.1,0,GTDB,isolate,Lactobacillus iners,FRT,GCF_000204435
576,GCF_000204435.2,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,GCF_000204435
4833,GCF_000214315.1,783244,ERR10897722.mbin.1,0,GTDB,isolate,Bifidobacterium vaginale,FRT,GCF_000214315
983,GCF_000214315.2,783244,ERR10897722.mbin.1,0,VMGC,MAG,Bifidobacterium vaginale,FRT,GCF_000214315


In [46]:
to_drop = df[df['genome_no_version'].isin(['GCF_000191685', 'GCF_000214315', 'GCF_000191705', 'GCF_000204435'])]
to_drop = to_drop[to_drop['database']=='GTDB']['genome'].tolist()

df = df[~df['genome'].isin(to_drop)]

In [47]:
df.value_counts(['species_name','source', 'genome_type']).sort_index()

species_name              source                  genome_type
Bifidobacterium vaginale  FRT                     MAG             685
                                                  isolate          28
                          other                   isolate           2
                          unknown                 isolate           1
                          urinary tract           isolate          16
Fannyhessea vaginae       FRT                     MAG             782
                                                  isolate           5
Lactobacillus crispatus   FRT                     MAG             812
                                                  isolate          58
                          gastrointestinal tract  isolate           6
                          non-human               isolate          50
                          oral                    isolate           2
                          other                   isolate           2
                          ur

In [48]:
def get_fasta_path(row):
    if row['database'] in ['VMGC', 'shared']:
        return f"/wynton/group/sirota/clairedubin/VMGC_db/VMGC_orig_files/VMGC_prokaryote_MAG/{row['genome']}.fa"
    elif row['database'] == 'GTDB':
        return f"/wynton/group/sirota/clairedubin/VMGC_GTDB_combined_db/GTDB_genomes/GTDB_genomes/{row['genome']}.fna"

# Apply the function row by row
df['fasta_path'] = df.apply(get_fasta_path, axis=1)

In [49]:
frt_mags = df[(df['source'] == 'FRT') & (df['genome_type']=='MAG')]
frt_isolates = df[(df['source'] == 'FRT') & (df['genome_type']=='isolate')]

frt_mags.value_counts('species_name')

species_name
Lactobacillus iners         1856
Lactobacillus crispatus      812
Fannyhessea vaginae          782
Bifidobacterium vaginale     685
Lactobacillus jensenii       321
Lactobacillus gasseri        195
Name: count, dtype: int64

In [50]:
frt_isolates[frt_isolates
             ['database'] == 'GTDB']

,genome,species,representative,genome_is_representative,database,genome_type,species_name,source,genome_no_version,fasta_path
4748,GCA_900452495.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_900452495,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4795,GCF_004361075.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361075,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4796,GCF_004361095.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361095,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4797,GCF_004361115.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361115,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4798,GCF_004361125.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361125,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4799,GCF_004361175.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361175,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4800,GCF_004361195.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361195,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4801,GCF_004361205.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361205,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4802,GCF_004361215.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361215,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4803,GCF_004361245.1,988598,GCF_000162255.1,0,GTDB,isolate,Lactobacillus crispatus,FRT,GCF_004361245,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...


In [51]:
frt_isolates[['genome', 'species', 'representative', 'genome_is_representative','fasta_path']].to_csv('rarefaction_inputs/FRT_isolates.tsv', index=False, sep='\t')
frt_mags[['genome', 'species', 'representative', 'genome_is_representative','fasta_path']].to_csv('rarefaction_inputs/FRT_MAGs.tsv', index=False, sep='\t')

In [ ]:
frt_counts = frt_isolates.value_counts('species').to_frame().reset_index()
frt_counts.columns=['species','starting_sample_size']
frt_counts.to_csv('rarefaction_inputs/FRT_starting_counts.csv', index=False)

In [54]:
frt_counts

,species,starting_sample_size
0,988598,58
1,783244,28
2,240891,20
3,611554,8
4,571325,6
5,619501,5


In [55]:
df[df['source'] == 'FRT']

,genome,species,representative,genome_is_representative,database,genome_type,species_name,source,genome_no_version,fasta_path
0,GCF_000439915.2,611554,GCF_000439915.2,1,shared,isolate,Lactobacillus gasseri,FRT,GCF_000439915,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...
1,SRR17284223.mbin.1,240891,SRR17284223.mbin.1,1,VMGC,MAG,Lactobacillus iners,FRT,SRR17284223.mbin.1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...
2,SRR10258542.mbin.1,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,SRR10258542.mbin.1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...
3,MG329.mbin.1,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,MG329.mbin.1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...
4,P10709414.mbin.1,240891,SRR17284223.mbin.1,0,VMGC,MAG,Lactobacillus iners,FRT,P10709414.mbin.1,/wynton/group/sirota/clairedubin/VMGC_db/VMGC_...
...,...,...,...,...,...,...,...,...,...,...
4856,GCF_004336685.1,783244,ERR10897722.mbin.1,0,GTDB,isolate,Bifidobacterium vaginale,FRT,GCF_004336685,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4863,GCF_900105405.1,783244,ERR10897722.mbin.1,0,GTDB,isolate,Bifidobacterium vaginale,FRT,GCF_900105405,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4873,GCF_002287905.1,611554,GCF_000439915.2,0,GTDB,isolate,Lactobacillus gasseri,FRT,GCF_002287905,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...
4889,GCF_000177755.1,240891,SRR17284223.mbin.1,0,GTDB,isolate,Lactobacillus iners,FRT,GCF_000177755,/wynton/group/sirota/clairedubin/VMGC_GTDB_com...


In [56]:
df[df['source'].isna()]

,genome,species,representative,genome_is_representative,database,genome_type,species_name,source,genome_no_version,fasta_path
